# PhishGuard – AI-Based Phishing URL Detection System
## AI/ML Mini Project (Experiment 12): Exploratory Data Analysis & Model Comparison

This notebook demonstrates the complete Machine Learning pipeline used in **PhishGuard**:
1. **Dataset Loading & Preprocessing** (`dataset/phishing_dataset.csv`)
2. **Deterministic URL Feature Extraction** (22 lexical, host, and entropy features)
3. **Feature Statistical Summary & Class Comparison**
4. **Model Training & Evaluation** (Logistic Regression, Decision Tree, Random Forest, SVM, XGBoost)
5. **Confusion Matrix, ROC-AUC, and Feature Importance Analysis**

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path so backend modules can be imported
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from backend.features.url_features import extract_features_vector, FEATURE_NAMES
print(f'Loaded {len(FEATURE_NAMES)} URL features:', FEATURE_NAMES)

### 1. Load & Inspect Dataset

In [ ]:
dataset_path = os.path.join(PROJECT_ROOT, 'dataset', 'phishing_dataset.csv')
df = pd.read_csv(dataset_path)
print(f'Dataset Shape: {df.shape}')
print('\nClass Distribution (0 = Legitimate, 1 = Phishing):')
print(df['label'].value_counts())
df.head(10)

### 2. Feature Extraction & Summary Statistics

In [ ]:
X_matrix = np.array([extract_features_vector(u) for u in df['url']], dtype=float)
features_df = pd.DataFrame(X_matrix, columns=FEATURE_NAMES)
features_df['label'] = df['label'].values

# Compare mean feature values between Legitimate (0) and Phishing (1)
features_df.groupby('label')[['url_length', 'hostname_length', 'num_dots', 'num_hyphens', 'has_https', 'suspicious_keyword_count', 'domain_entropy']].mean().T

### 3. Model Evaluation Metrics Comparison

In [ ]:
metrics_path = os.path.join(PROJECT_ROOT, 'backend', 'models', 'metrics.json')
with open(metrics_path, 'r') as f:
    metrics_data = json.load(f)

print('Selected Best Model:', metrics_data['best_model'])
comparison_df = pd.DataFrame([
    {
        'Model': m['model_name'],
        'Accuracy': m['accuracy'],
        'Precision': m['precision'],
        'Recall': m['recall'],
        'F1-Score': m['f1_score'],
        'ROC-AUC': m['roc_auc']
    }
    for m in metrics_data['models']
]).sort_values(by='F1-Score', ascending=False)

comparison_df